# FASE 2: PRÉ-PROCESSAMENTO E IMPLEMENTAÇÃO DO BASELINE

In [14]:
import pandas as pd
import numpy as np
import zipfile
import glob

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


In [15]:
# Descompactar o arquivo zipado se necessário e carregar o CSV
with zipfile.ZipFile('salary.csv (1).zip', 'r') as zip_ref:
    zip_ref.extractall('/content')

arquivo_csv = glob.glob('/content/*.csv')[0]
df = pd.read_csv(arquivo_csv)

print(f"Arquivo carregado! Total de linhas: {df.shape[0]}")
display(df.head())

Arquivo carregado! Total de linhas: 32561


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


# 1. PRÉ-PROCESSAMENTO E TRATAMENTO DOS DADOS

In [16]:
# Tratamento de valores ausentes ('?') e remoção da coluna 'fnlwgt'
df = df.replace('?', np.nan)
df_model = df.drop(columns=['fnlwgt'], errors='ignore')

# Separação entre variáveis preditoras (X) e a coluna alvo correta ('salary')
X = df_model.drop(columns=['salary'])
y = df_model['salary'].astype(str).str.strip().apply(lambda x: 1 if '>50K' in x else 0)

# Identificação de colunas numéricas e categóricas
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'string']).columns.tolist()

# Configuração dos pipelines de tratamento
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ]
)

# Divisão estratificada em Treino (80%) e Teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dados pré-processados e divididos com sucesso!")
print(f"-> Amostras de Treino: {X_train.shape[0]}")
print(f"-> Amostras de Teste:  {X_test.shape[0]}")

Dados pré-processados e divididos com sucesso!
-> Amostras de Treino: 26048
-> Amostras de Teste:  6513


#Treinamento e Avaliação do Baseline

In [17]:
# Definição dos modelos de baseline (Regressão Logística e Árvore de Decisão)
models = {
    "Regressão Logística": LogisticRegression(random_state=42, max_iter=1000),
    "Árvore de Decisão": DecisionTreeClassifier(random_state=42)
}

results = []

print("="*60)
print("TREINAMENTO E AVALIAÇÃO DOS MODELOS BASELINE")
print("="*60 + "\n")

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    results.append({
        "Algoritmo": name,
        "Acurácia": acc,
        "Acurácia Balanceada": bal_acc,
        "Precisão": precision,
        "Revocação (Recall)": recall,
        "F1-Score": f1
    })

    print(f"=== Relatório de Classificação: {name} ===")
    print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))
    print("-" * 60 + "\n")

# Tabela resumo comparativa final
df_results = pd.DataFrame(results)
print("=== TABELA RESUMO COMPARATIVA ===")
display(df_results.style.format({
    "Acurácia": "{:.4f}",
    "Acurácia Balanceada": "{:.4f}",
    "Precisão": "{:.4f}",
    "Revocação (Recall)": "{:.4f}",
    "F1-Score": "{:.4f}"
}))

TREINAMENTO E AVALIAÇÃO DOS MODELOS BASELINE

=== Relatório de Classificação: Regressão Logística ===
              precision    recall  f1-score   support

       <=50K       0.88      0.93      0.91      4945
        >50K       0.74      0.62      0.67      1568

    accuracy                           0.85      6513
   macro avg       0.81      0.77      0.79      6513
weighted avg       0.85      0.85      0.85      6513

------------------------------------------------------------

=== Relatório de Classificação: Árvore de Decisão ===
              precision    recall  f1-score   support

       <=50K       0.88      0.89      0.89      4945
        >50K       0.65      0.63      0.64      1568

    accuracy                           0.83      6513
   macro avg       0.77      0.76      0.76      6513
weighted avg       0.83      0.83      0.83      6513

------------------------------------------------------------

=== TABELA RESUMO COMPARATIVA ===


,Algoritmo,Acurácia,Acurácia Balanceada,Precisão,Revocação (Recall),F1-Score
0,Regressão Logística,0.8544,0.7733,0.7359,0.6167,0.6711
1,Árvore de Decisão,0.8279,0.7590,0.6473,0.6263,0.6366
